# Notebook de Validation - Pipeline ASR Lingala (WAXAL Dataset)

Ce notebook permet de tester le pipeline de nettoyage sur **100 échantillons** et de visualiser les métadonnées et la qualité des données audio/textuelles générées.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import json
from IPython.display import display 

# Ajouter le projet au chemin système
sys.path.insert(0, str(Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())))

from config.settings import get_settings
from src.pipeline import Pipeline

## 1. Exécution du Pipeline sur 100 Échantillons

In [ ]:
test_output_dir = "./data_notebook_test"

# Configuration pour le test rapide
settings = get_settings(
    output_dir=test_output_dir,
    hf_dataset_name="google/waxal",
    hf_config_name="lin",
    num_workers=2,
    batch_size=20
)

pipeline = Pipeline(settings)
print("Lancement du pipeline...")
pipeline.run(max_samples=100, resume=False)

## 2. Inspection du Manifest Résumé

In [ ]:
manifest_path = Path(test_output_dir) / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(json.dumps(manifest, indent=2, ensure_ascii=False))
else:
    print("Fichier manifest non trouvé.")

## 3. Visualisation des Données Parquet (Split Train)

In [ ]:
train_parquet = Path(test_output_dir) / "train.parquet"
if train_parquet.exists():
    df_train = pd.read_parquet(train_parquet)
    print(f"Dimensions du dataset d'entraînement : {df_train.shape}")
    display(df_train.head())
else:
    print("Fichier train.parquet non disponible.")

## 4. Statistiques SNR et Durées Audio

In [ ]:
if 'df_train' in locals():
    print("Statistiques descriptives des segments :")
    display(df_train[['duration', 'snr_db', 'vad_ratio', 'word_count']].describe())

## 5. Distribution des Durées des Segments

Tracé de l'histogramme des durées des segments audio générés pour s'assurer qu'ils sont bien compris entre 2 et 25 secondes (majoritairement de 3 à 15 secondes).

In [ ]:
import matplotlib.pyplot as plt
if 'df_train' in locals() and not df_train.empty:
    plt.figure(figsize=(10, 5))
    plt.hist(df_train['duration'], bins=20, color='#3498db', edgecolor='black', alpha=0.8)
    plt.title('Distribution des durées des segments générés', fontsize=14)
    plt.xlabel('Durée (secondes)', fontsize=12)
    plt.ylabel('Nombre de segments', fontsize=12)
    plt.axvline(2.0, color='red', linestyle='--', linewidth=1.5, label='Min absolu (2s)')
    plt.axvline(3.0, color='orange', linestyle='--', linewidth=1.5, label='Min cible (3s)')
    plt.axvline(15.0, color='green', linestyle='--', linewidth=1.5, label='Max cible (15s)')
    plt.axvline(25.0, color='red', linestyle='--', linewidth=1.5, label='Max absolu (25s)')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
else:
    print("Aucun DataFrame df_train disponible pour l'affichage de l'histogramme.")